## Reprojecting a LC for London

| Use Case                  | Best Projection          | EPSG Code |
|---------------------------|-------------------------|-----------|
| UK-specific studies       | British National Grid   | 27700     |
| General mapping (London)  | UTM Zone 30N            | 32630     |
| Europe-wide studies       | ETRS89 LAEA Europe      | 3035      |


In [ ]:
import os
current_dir = os.getcwd()
print(current_dir)


g:\Shared drives\Wellcome Trust Project Data


In [2]:

from rasterio.warp import calculate_default_transform, reproject, Resampling
import rasterio

# Define the target projected CRS (British National Grid)
target_crs = "EPSG:27700"

# Construct the file path
file_path = os.path.join(current_dir, "1_preprocess", "ESA_WorldCover_10m_2021_v200_Mosaic_Mask.tif")
file_out = os.path.join(current_dir, "1_preprocess", "ESA_WorldCover_10m_2021_v200_Mosaic_Mask_proj.tif")

with rasterio.open(file_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, target_crs, src.width, src.height, *src.bounds
    )
    
    kwargs = src.meta.copy()
    kwargs.update({"crs": target_crs, "transform": transform, "width": width, "height": height})

    with rasterio.open(file_out, "w", **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=Resampling.nearest
            )

print("Reprojection complete. Saved as: " + os.path.basename(file_out))


Reprojection complete. Saved as: ESA_WorldCover_10m_2021_v200_Mosaic_Mask_proj.tif
